In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
cwd = Path.cwd()
project_root = cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

Done!


In [3]:
corona_effect = pd.read_csv(project_root / 'data' / 'online_classroom.csv')
corona_effect.head()

,gender,asian,black,hawaiian,hispanic,unknown,white,format_ol,format_blended,falsexam
0,0,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0,63.29997
1,1,0.0,0.0,0.0,0.0,0.0,1.0,0,0.0,79.96000
2,1,0.0,0.0,0.0,0.0,0.0,1.0,0,1.0,83.37000
3,1,0.0,0.0,0.0,0.0,0.0,1.0,0,1.0,90.01994
4,1,0.0,0.0,0.0,0.0,0.0,1.0,1,0.0,83.30000


In [4]:
causal_effect = (corona_effect.assign(
    class_format=np.select(
                [corona_effect['format_ol'].astype(bool), corona_effect['format_blended'].astype(bool)],
                ['online', 'blended'],
                default="face_to_face"
    )
)
.groupby('class_format')['falsexam']
.mean()
)

In [5]:
causal_effect

class_format
blended         77.093731
face_to_face    78.547485
online          73.635263
Name: falsexam, dtype: float64

Yup. It’s that simple. We can see that face-to-face classes yield a 78.54 average score, while online courses yield a 73.63 average score. Not so good news for the proponents of online learning. The 
 for an online class is thus -4.91. This means that online classes cause students to perform about 5 points lower, on average. That’s it. You don’t need to worry that online courses might have poorer students that can’t afford face-to-face classes or, for that matter, you don’t have to worry that the students from the different treatments are different in any way other than the treatment they received. By design, the random experiment is made to wipe out those differences.

In [6]:
(corona_effect.assign(
    class_format=np.select(
                [corona_effect['format_ol'].astype(bool), corona_effect['format_blended'].astype(bool)],
                ['online', 'blended'],
                default="face_to_face"
    )
)
.groupby('class_format')
.mean()
)

,gender,asian,black,hawaiian,hispanic,unknown,white,format_ol,format_blended,falsexam
class_format,,,,,,,,,,
blended,0.550459,0.217949,0.102564,0.025641,0.012821,0.012821,0.628205,0.0,1.0,77.093731
face_to_face,0.633333,0.202020,0.070707,0.000000,0.010101,0.000000,0.717172,0.0,0.0,78.547485
online,0.542553,0.228571,0.028571,0.014286,0.028571,0.000000,0.700000,1.0,0.0,73.635263


For this reason, a good sanity check to see if the randomisation was done right (or if you are looking at the correct data) is to check if the treated are equal to the untreated in pre-treatment variables. Our data has information on gender and ethnicity to see if they are similar across groups. We can say that they look pretty similar for the gender, asian, hispanic, and white variables. The black variable, however, seems a little bit different. This draws attention to what happens with a small dataset. Even under randomisation, it could be that, by chance, one group is different from another. In large samples, this difference tends to disappear.

Let's Check if this impact is statistically significant

In [7]:
corona_effect.info()

<class 'pandas.DataFrame'>
RangeIndex: 323 entries, 0 to 322
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   gender          323 non-null    int64  
 1   asian           247 non-null    float64
 2   black           247 non-null    float64
 3   hawaiian        247 non-null    float64
 4   hispanic        247 non-null    float64
 5   unknown         247 non-null    float64
 6   white           247 non-null    float64
 7   format_ol       323 non-null    int64  
 8   format_blended  323 non-null    float64
 9   falsexam        323 non-null    float64
dtypes: float64(8), int64(2)
memory usage: 25.4 KB


In [8]:
online = corona_effect.loc[corona_effect['format_ol'].astype(bool) == True]['falsexam']
face_to_face = corona_effect.loc[(corona_effect['format_ol'].astype(bool) == False) & (corona_effect['format_blended'].astype(bool) == False)]['falsexam']

In [9]:
def SE(y: pd.Series) -> float:
    return (np.std(y) / np.sqrt(len(y))).item()

In [10]:
print(f'Online: {SE(online):.3f}')
print(f'Face_to_Face: {SE(face_to_face):.3f}')


Online: 1.529
Face_to_Face: 0.869


In [11]:
def ci(y: pd.Series) -> float:
    return ((y.mean() - 2*(y.std()/len(y)**0.5)).item(), (y.mean() + 2*(y.std()/len(y)**0.5)).item())

In [12]:
print(f"95% CI Online Learning: {ci(online)}")
print(f"95% CI Face yo Face Learning: {ci(face_to_face)}")

95% CI Online Learning: (70.56094429049804, 76.7095818797147)
95% CI Face yo Face Learning: (76.80278229206951, 80.29218687459715)


We can see that the 95% CIs of the groups don’t overlap. The lower end of the CI for Face to Face class is above the upper end of the CI for online classes. This is evidence that our result is not by chance and that the true mean for students in face-to-face classes is higher than the true mean for students in online classes. In other words, there is a significant causal decrease in academic performance when switching from face-to-face to online classes.

Let’s return to our classroom example. We will construct this distribution of the difference. Of course, once we have it, building the 95% CI is straightforward.

In [16]:
diff_mu = online.mean() - face_to_face.mean()
diff_se = np.sqrt(online.var()/len(online) + face_to_face.var()/len(face_to_face))

ci = ((diff_mu - 2 * diff_se).item(), (diff_mu + 2 * diff_se).item())

In [17]:
ci

(-8.44710793714168, -1.3773350593122302)

With this at hand, we can say that we are 95% confident that the true difference between the online and face-to-face groups falls between -8.37 and -1.44. We can also construct a z statistic by dividing the difference in mean by the 
 of the differences.

In [19]:
z_statistic = diff_mu / diff_se
z_statistic.item()

-2.7792810791031224

This looks like a pretty extreme value. Indeed, it is above 2, which means there is less than a 5% chance that we would see such an extreme value if there were no difference in the groups. This again leads us to conclude that switching from face-to-face to online classes causes a statistically significant drop in academic performance.